# Chatbot de Autos con RAG (LangChain + OpenAI + FAISS)

Chatbot que permite hacer búsquedas semánticas sobre un dataset de autos usando:
- RAG (Retrieval Augmented Generation)
- FAISS - Vector store rápido y local
- OpenAI Embeddings - Para vectorizar los datos
- LangChain - Orquestación simple


!pip install langchain langchain-openai langchain-community langchain-core faiss-cpu pandas --quiet

import os
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-core faiss-cpu pandas --quiet

In [ ]:
import os
import pandas as pd
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')


## 3. Configurar API Key

Obtené tu API key en: https://platform.openai.com/api-keys

In [ ]:
import os

# To get your API key: https://platform.openai.com/api-keys
# In Google Colab: click the key icon in the sidebar, add a secret called OPENAI_API_KEY
# Then run this cell to load it:

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API Key loaded successfully")

## 4. Cargar dataset

In [ ]:
# Update this path to where your dataset is located
# The dataset should be a CSV with columns: manufacturer_name, model_name, year_produced,
# price_usd, engine_fuel, engine_capacity, transmission, odometer_value,
# body_type, color, drivetrain, state
DATASET_PATH = "cars.csv"  # Update this to your file path

df = pd.read_csv(DATASET_PATH)
print(f"📊 Dataset loaded: {df.shape[0]} cars, {df.shape[1]} columns")
print(f"
Columns: {list(df.columns)}")
df.head()

## 5. Convertir datos a formato narrativo

Cada auto se convierte en una descripción en texto natural para mejorar la búsqueda semántica.

In [ ]:
def row_to_text(row):
    """
    Convierte una fila del DataFrame a texto descriptivo narrativo.
    """
    # Manejar valores nulos
    def safe_val(val, default="desconocido"):
        return val if pd.notna(val) else default

    text = f"""{safe_val(row['manufacturer_name'])} {safe_val(row['model_name'])} del año {safe_val(row['year_produced'], 'N/A')}.
Precio: ${safe_val(row['price_usd'], 0):,.0f} USD.
Motor: {safe_val(row['engine_fuel'])} de {safe_val(row['engine_capacity'], 'N/A')} litros.
Transmisión: {safe_val(row['transmission'])}.
Kilometraje: {safe_val(row['odometer_value'], 0):,.0f} km.
Tipo de carrocería: {safe_val(row['body_type'])}.
Color: {safe_val(row['color'])}.
Tracción: {safe_val(row['drivetrain'])}.
Estado: {safe_val(row['state'])}.
"""
    return text.strip()

# Probar con el primer auto
print("Ejemplo de descripción narrativa:\n")
print(row_to_text(df.iloc[0]))
print("\n✅ Función de conversión creada")

## 6. Crear documentos para vectorización

Convertimos cada auto a un `Document` de LangChain con texto + metadata.

In [ ]:
documents = []

for idx, row in df.iterrows():
    # Crear texto narrativo
    text = row_to_text(row)

    # Crear metadata con info estructurada
    metadata = {
        "manufacturer": str(row['manufacturer_name']),
        "model": str(row['model_name']),
        "year": str(row['year_produced']),
        "price": float(row['price_usd']) if pd.notna(row['price_usd']) else 0,
        "color": str(row['color']),
        "body_type": str(row['body_type']),
        "engine_fuel": str(row['engine_fuel']),
        "index": idx
    }

    # Crear documento
    doc = Document(page_content=text, metadata=metadata)
    documents.append(doc)

print(f"✅ {len(documents)} documentos creados")
print(f"\nEjemplo del primer documento:")
print(f"Texto: {documents[0].page_content[:200]}...")
print(f"Metadata: {documents[0].metadata}")

## 7. Crear embeddings y vector store con FAISS

**Nota:** Este paso tarda un poco la primera vez (genera embeddings para todos los autos).
Después se guarda en disco y se carga instantáneamente.

In [ ]:
import os.path

VECTOR_STORE_PATH = "./faiss_autos_index"

# Inicializar embeddings de OpenAI
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # Modelo más económico y rápido
)

# Verificar si ya existe el vector store
if os.path.exists(VECTOR_STORE_PATH):
    print("📂 Cargando vector store existente desde disco...")
    vectorstore = FAISS.load_local(
        VECTOR_STORE_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Vector store cargado desde disco")
else:
    print("🔄 Creando vector store por primera vez (esto puede tardar 1-2 minutos)...")
    vectorstore = FAISS.from_documents(documents, embeddings)

    # Guardar en disco para uso futuro
    vectorstore.save_local(VECTOR_STORE_PATH)
    print(f"✅ Vector store creado y guardado en '{VECTOR_STORE_PATH}'")

print(f"\n📊 Total de vectores en el store: {vectorstore.index.ntotal}")

## 8. Configurar el LLM y crear el chain de RetrievalQA

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Inicializar modelo de OpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,  # Respuestas más determinísticas
)

# Crear retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Template personalizado en español
template = """Sos un asistente experto en autos que ayuda a los usuarios a encontrar información sobre vehículos.

Usa el siguiente contexto para responder la pregunta del usuario. El contexto contiene información sobre autos reales del dataset.

Contexto:
{context}

Pregunta: {question}

Instrucciones:
- Respondé en español de forma clara y concisa
- Si encontrás varios autos relevantes, mencioná los más destacados (máximo 5)
- Incluí detalles específicos como marca, modelo, año, precio cuando sea relevante
- Si no hay información suficiente en el contexto, decí que no encontraste autos que coincidan
- Podés hacer comparaciones si el usuario lo pide

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# Función para formatear documentos
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Crear el chain RAG usando LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Chain RAG creado")

## 9. Función para hacer preguntas

In [ ]:
# Inicializar historial de conversación
historial_conversacion = []

def preguntar(pregunta: str, mostrar_fuentes: bool = False):
    """
    Hace una pregunta al chatbot usando RAG con memoria de conversación.

    Args:
        pregunta: La pregunta en lenguaje natural
        mostrar_fuentes: Si True, muestra los autos usados como fuente

    Returns:
        La respuesta del chatbot
    """
    try:
        # 1. Obtener documentos relevantes
        docs = retriever.invoke(pregunta)

        # 2. Mostrar fuentes si se solicita
        if mostrar_fuentes:
            print("\n" + "="*60)
            print("📚 FUENTES USADAS (Top 5 autos más relevantes):")
            print("="*60)
            for i, doc in enumerate(docs, 1):
                print(f"\n{i}. {doc.metadata['manufacturer']} {doc.metadata['model']} ({doc.metadata['year']})")
                print(f"   Precio: ${doc.metadata['price']:,.0f} USD")
                print(f"   Color: {doc.metadata['color']} | Tipo: {doc.metadata['body_type']}")
            print("="*60 + "\n")

        # 3. Formatear el contexto
        context = "\n\n".join([doc.page_content for doc in docs])

        # 4. Formatear el historial de conversación
        historial_texto = ""
        if historial_conversacion:
            historial_texto = "\n\nHistorial de conversación previa:\n"
            for msg in historial_conversacion[-6:]:  # Solo las últimas 3 interacciones (6 mensajes)
                historial_texto += f"{msg['role']}: {msg['content']}\n"

        # 5. Crear el prompt con historial
        prompt_text = f"""Sos un asistente experto en autos que ayuda a los usuarios a encontrar información sobre vehículos.

Usa el siguiente contexto para responder la pregunta del usuario. El contexto contiene información sobre autos reales del dataset.

Contexto:
{context}
{historial_texto}

Pregunta actual: {pregunta}

Instrucciones:
- Respondé en español de forma clara y concisa
- Si el usuario hace referencia a preguntas anteriores, usa el historial para mantener contexto
- Si encontrás varios autos relevantes, mencioná los más destacados (máximo 5)
- Incluí detalles específicos como marca, modelo, año, precio cuando sea relevante
- Si no hay información suficiente en el contexto, decí que no encontraste autos que coincidan
- Podés hacer comparaciones si el usuario lo pide

Respuesta:"""

        # 6. Invocar el LLM directamente
        response = llm.invoke(prompt_text)

        # 7. Guardar la interacción en el historial
        historial_conversacion.append({
            "role": "Usuario",
            "content": pregunta
        })
        historial_conversacion.append({
            "role": "Asistente",
            "content": response.content
        })

        # 8. Extraer el contenido de la respuesta
        return response.content

    except Exception as e:
        import traceback
        error_detail = traceback.format_exc()
        return f"❌ Error: {str(e)}\n\nDetalles:\n{error_detail}"

def limpiar_historial():
    """Limpia el historial de conversación"""
    global historial_conversacion
    historial_conversacion = []
    print("🧹 Historial limpiado")

print("✅ Función de consulta creada con memoria (sin LCEL)")

## 10. Ejemplos de búsquedas semánticas

Probá diferentes tipos de preguntas:

In [ ]:
# Limpiar historial para empezar de cero
limpiar_historial()

# Primera pregunta
print("=" * 60)
print("🧑 Pregunta 1: Muéstrame SUVs Toyota")
print("=" * 60)
respuesta1 = preguntar("Muéstrame SUVs Toyota")
print(f"\n🤖 Respuesta:\n{respuesta1}")

# Segunda pregunta que hace referencia a la anterior
print("\n\n" + "=" * 60)
print("🧑 Pregunta 2: ¿Cuál es el más barato de esos?")
print("=" * 60)
print("(Nota: Esta pregunta hace referencia a los autos de la pregunta anterior)")
respuesta2 = preguntar("¿Cuál es el más barato de esos?")
print(f"\n🤖 Respuesta:\n{respuesta2}")

# Tercera pregunta para seguir el contexto
print("\n\n" + "=" * 60)
print("🧑 Pregunta 3: ¿Y tiene garantía?")
print("=" * 60)
print("(Nota: Esta pregunta hace referencia al auto más barato mencionado)")
respuesta3 = preguntar("¿Y tiene garantía?")
print(f"\n🤖 Respuesta:\n{respuesta3}")

print("\n\n" + "=" * 60)
print("✅ Como ves, el chatbot recuerda el contexto de la conversación!")
print("=" * 60)

In [ ]:
# Ejemplo 1: Búsqueda por características
print("\n" + "="*60)
print("❓ Pregunta: Muéstrame autos SUV que sean económicos")
print("="*60)
respuesta = preguntar("Muéstrame autos SUV que sean económicos", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")

In [ ]:
# Ejemplo 4: Búsqueda por precio
print("\n" + "="*60)
print("❓ Pregunta: Necesito un auto familiar por menos de $10,000")
print("="*60)
respuesta = preguntar("Necesito un auto familiar por menos de $10,000", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")

In [ ]:
# Ejemplo 5: Comparación
print("\n" + "="*60)
print("❓ Pregunta: Compará autos Honda vs Mazda")
print("="*60)
respuesta = preguntar("Compará autos Honda vs Mazda", mostrar_fuentes=True)
print(f"\n🤖 Respuesta:\n{respuesta}")

## 11. Chatbot interactivo

Escribí tus preguntas en lenguaje natural.

**Comandos:**
- `salir` - Terminar el chat
- `fuentes` - Activar/desactivar mostrar fuentes

In [ ]:
print("=" * 60)
print("🚗 Chatbot de Autos con RAG + Memoria")
print("Preguntame sobre autos en lenguaje natural.")
print("")
print("Comandos: 'salir', 'fuentes' (activar/desactivar), 'limpiar' (borrar historial)")
print("=" * 60)

mostrar_fuentes = False
print(f"\n💡 Fuentes: {'Activadas' if mostrar_fuentes else 'Desactivadas'} (escribe 'fuentes' para cambiar)")
print(f"📚 Historial: Activo (escribe 'limpiar' para reiniciar la conversación)\n")

while True:
    pregunta_usuario = input("\n🧑 Vos: ")

    if pregunta_usuario.strip().lower() == "salir":
        print("\n¡Chau! 👋")
        break

    elif pregunta_usuario.strip().lower() == "fuentes":
        mostrar_fuentes = not mostrar_fuentes
        print(f"\n💡 Fuentes: {'Activadas ✅' if mostrar_fuentes else 'Desactivadas ❌'}")
        continue

    elif pregunta_usuario.strip().lower() == "limpiar":
        limpiar_historial()
        continue

    if pregunta_usuario.strip():
        try:
            respuesta = preguntar(pregunta_usuario, mostrar_fuentes=mostrar_fuentes)
            print(f"\n🤖 Bot: {respuesta}")
        except Exception as e:
            print(f"\n❌ Error: {e}")

## 12. Búsqueda directa en el vector store (sin LLM)

También podés hacer búsquedas directas para ver los autos más similares sin generar respuesta con el LLM.

In [ ]:
def busqueda_directa(query: str, k: int = 5):
    """
    Búsqueda directa por similitud sin usar el LLM.
    """
    docs = vectorstore.similarity_search(query, k=k)

    print(f"\n🔍 Top {k} autos más similares a: '{query}'\n")
    print("="*60)

    for i, doc in enumerate(docs, 1):
        print(f"\n{i}. {doc.metadata['manufacturer']} {doc.metadata['model']} ({doc.metadata['year']})")
        print(f"   💰 Precio: ${doc.metadata['price']:,.0f} USD")
        print(f"   🎨 Color: {doc.metadata['color']}")
        print(f"   🚗 Tipo: {doc.metadata['body_type']}")
        print(f"   ⛽ Combustible: {doc.metadata['engine_fuel']}")

    print("\n" + "="*60)

# Ejemplo
busqueda_directa("sedán azul automático económico", k=5)